In [1018]:
# 설치
# !pip install -qU pypdf

In [1019]:
# OCR
# !pip install -qU rapidocr-onnxruntime

In [1020]:
# from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [1021]:
# !pip install pdfminer.six

In [1022]:
# !pip install langchain-community pymupdf

In [1023]:
# !pip install -qU docx2txt

In [1024]:
# !pip install unstructured python-pptx

In [1025]:
# import nltk
# nltk.download('punkt_tab')

In [1026]:
# Reference
# https://m.blog.naver.com/htk1019/223442628204

In [1027]:
import bs4

In [1028]:
from langchain_core.documents import Document

In [1029]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [1030]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, PDFMinerLoader, UnstructuredPDFLoader
from langchain_community.document_loaders import Docx2txtLoader, UnstructuredPowerPointLoader, TextLoader
# from langchain_community.document_loaders import WebBaseLoader

In [1031]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [1032]:
import re
import json
import warnings
warnings.filterwarnings("ignore")

In [1033]:
all_raw_text = []

In [1034]:
def transform_clean_string(to_replace): 
    """remove unnecessary characters"""

    if isinstance(to_replace, (str)):
        to_replace = to_replace.strip()
        to_replace = re.sub(r'(?<!\.)(\n|\r\n)', ' ', to_replace)
        to_replace = re.sub(r'\t|\\t', ' ', to_replace)
        to_replace = re.sub(r' +', ' ', to_replace)

    # merge all texts
    all_raw_text.append(to_replace)
    
    return to_replace

In [1035]:
# --
# PDF Loader
# --
filepath = './files/asiabrief_3-26.pdf'

loader = PyPDFLoader(filepath, mode="page", images_inner_format="markdown-img", images_parser=RapidOCRBlobParser())
# loader = PyPDFDirectoryLoader("data/")
# loader = PDFMinerLoader(pdf_filepath)
# loader = UnstructuredPDFLoader(pdf_filepath, mode="elements")

# data = [Document(page_content=transform_trim_string(text.page_content), metadata=text.metadata) for i, text in enumerate(loader.load())]
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1036]:
# --
# DOC Loader
# --
filepath = './files/Sample.docx'

all_raw_text = []

loader = Docx2txtLoader(filepath)
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1037]:
# --
# DOC Loader
# --
filepath = './files/test.log'

all_raw_text = []

loader = TextLoader(filepath, encoding="utf-8")
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [1038]:
print(f'총 분할된 도큐먼트 수: {len(data)}')

총 분할된 도큐먼트 수: 1


In [1039]:
# data

In [1040]:
# print(f"전체 텍스트 추출: {''.join(all_raw_text)}")

In [1041]:
# 2. Split documents into small, manageable chunks
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0)
text_splitter = CharacterTextSplitter(chunk_size=3000, chunk_overlap=500, separator = '\n\n')
docs = text_splitter.split_documents(data)

In [1042]:
print(f'TextSplitter 후, 총 분할된 도큐먼트 수: {len(docs)}')

TextSplitter 후, 총 분할된 도큐먼트 수: 1


In [1043]:
# docs

In [1044]:
# -- Directory

In [1045]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

# 확장자별 로더 매핑
loader_mapping = {
    '.log': TextLoader,
    '.pdf': PyPDFLoader,
}

# 매핑된 로더를 loader_cls에 지정
# glob='./*.*'
# loader = DirectoryLoader('./files', glob="**/*.pdf", loader_cls=PyPDFLoader)
# loader = DirectoryLoader(path='./files', glob=['**/*.txt', '**/*.log'])
# docs = loader.load_and_split()

# 1. 각 확장자별로 DirectoryLoader 생성
txt_loader1 = DirectoryLoader('./files', glob='**/*.txt', loader_cls=TextLoader)
txt_loader2 = DirectoryLoader('./files', glob='**/*.log', loader_cls=TextLoader)
pdf_loader = DirectoryLoader('./files', glob='**/*.pdf', loader_cls=PyPDFLoader)

# 2. 문서 로드
txt_docs1 = txt_loader1.load()
txt_docs2 = txt_loader2.load()
pdf_docs = pdf_loader.load()

# 3. 문서 리스트 병합
docs = txt_docs1 + txt_docs2 + pdf_docs

In [1046]:
print(f"총 {len(docs)}개의 문서를 불러왔습니다.")

총 6개의 문서를 불러왔습니다.


In [1047]:
for s in docs:
    print(s.metadata.get("source"))

files/test.log
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf


In [1048]:
docs = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in docs]

In [1049]:
# docs

In [1050]:
docs[0].page_content

'24/10/11 12:35:22 INFO myLogger: 2024-10-11 12:35:22.075 DB Execute for TableName : Test 24/10/11 12:35:38 ERROR Executor: Exception in task 1.98 in stage 0.0 (TID 107) java.lang.OutOfMemoryError: Java heap space at java.util.Arrays.copyOf(Arrays.java:3332) at java.lang.AbstractStringBuilder.ensureCapacityInternal(AbstractStringBuilder.java:124) at java.lang.AbstractStringBuilder.append(AbstractStringBuilder.java:596) at java.lang.StringBuffer.append(StringBuffer.java:367) at java.io.StringWriter.write(StringWriter.java:94) at oracle.jdbc.driver.ClobAccessor.getStringNoPrefetch(ClobAccessor.java:479) at oracle.jdbc.driver.ClobAccessor.getString(ClobAccessor.java:456) at oracle.jdbc.driver.OracleCallableStatement.getString(OracleCallableStatement.java:652) at oracle.jdbc.driver.OracleCallableStatementWrapper.getString(OracleCallableStatementWrapper.java:860) at org.apache.commons.dbcp2.DelegatingCallableStatement.getString(DelegatingCallableStatement.java:87) 24/10/11 12:35:38 ERROR Sp